In [1]:
import numpy as np
import tensorflow as tf
import gene_data
import configs
import hnn

2024-12-03 21:42:43.489502: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1733233363.510618 3818967 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1733233363.517371 3818967 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-12-03 21:42:43.541232: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [4]:
import importlib
importlib.reload(hnn)
importlib.reload(configs)

config = configs.CONFIGS["LHNN_1DGaussianmixture"] # load the config which records all experiment parameters.

In [33]:
config

{'seed': 0,
 'load': True,
 'path': '/home/ycui/Documents/TSRLproject_JPMorgan/data/1DGaussianmixture.obj',
 'num_samples': 20,
 'per_train': 0.8,
 'input_dim': 2,
 'dist_name': '1D_Gauss_mix',
 'dt': 0.05,
 'num_lf': 400,
 'num_hidden': 100,
 'num_layers': 3,
 'output_dim': 2,
 'acti': 'tanh',
 'baseline': False,
 'field_type': 'solenoidal',
 'separate_fields': False,
 'train_epoch': 30,
 'train_step': 200,
 'path_model': '/home/ycui/Documents/TSRLproject_JPMorgan/models/1DGaussianmixture.keras'}

In [4]:
np.random.seed(config["seed"])
tf.random.set_seed(config["seed"])

data = gene_data.get_dataset(**config)
# arrange data
train_states = tf.convert_to_tensor( data['train_states'], dtype=tf.float32)
test_states = tf.convert_to_tensor(data['test_states'], dtype=tf.float32)
train_timegrads = tf.convert_to_tensor(data['train_timegrads'], dtype=tf.float32)
test_timegrads = tf.convert_to_tensor(data['test_timegrads'], dtype=tf.float32)



Successfully loaded data


I0000 00:00:1733213103.711369 3795765 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 517 MB memory:  -> device: 0, name: Tesla V100-PCIE-16GB, pci bus id: 0000:3b:00.0, compute capability: 7.0
I0000 00:00:1733213103.712409 3795765 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 14478 MB memory:  -> device: 1, name: Tesla V100-PCIE-16GB, pci bus id: 0000:af:00.0, compute capability: 7.0
I0000 00:00:1733213103.712902 3795765 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:2 with 14478 MB memory:  -> device: 2, name: Tesla V100-PCIE-16GB, pci bus id: 0000:d8:00.0, compute capability: 7.0


In [28]:
nn = hnn.MLP(config["input_dim"], config["num_hidden"], config["num_layers"], config["output_dim"], config["acti"])
model = hnn.HNN(config["input_dim"], nn, baseline=config["baseline"], field_type=config["field_type"])

loss_obj = tf.keras.losses.MeanSquaredError()
optimizer = tf.keras.optimizers.Adam()

train_loss = tf.keras.metrics.Mean(name = "train_loss")
test_loss = tf.keras.metrics.Mean(name = "test_loss")

@tf.function
def train_step(x,y):
    with tf.GradientTape() as tape:
        y_pred = model.get_gradient(x, separate_fields = config["separate_fields"])
        loss = loss_obj(y,y_pred)
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    train_loss(loss)

@tf.function
def test_step(x,y):
    y_pred = model.get_gradient(x, separate_fields = config["separate_fields"])
    loss = loss_obj(y,y_pred)
    test_loss(loss)

In [29]:
for idx_epoch in range(config["train_epoch"]):
    print(f"Begin the {idx_epoch}th training epoch:")
    for idx_step in range(config["train_step"]):
        idxs_shuffle = tf.random.shuffle(tf.range(train_states.shape[0]))
        shuffled_states, shuffled_timegrads = tf.gather(train_states, idxs_shuffle), tf.gather(train_timegrads, idxs_shuffle)
        train_step(shuffled_states, shuffled_timegrads)
        test_step(test_states, test_timegrads)
    

    train_timegrads_hat = model.get_gradient(train_states)
    train_dist = (train_timegrads - train_timegrads_hat)**2 
    test_timegrads_hat = model.get_gradient(test_states)
    test_dist = (test_timegrads - test_timegrads_hat)**2
    print('Train loss {:.4e} +/- {:.4e}\n test loss {:.4e} +/- {:.4e}'
        .format(tf.reduce_mean(train_dist).numpy(), tf.math.reduce_std(train_dist).numpy()/np.sqrt(train_dist.shape[0]),
                tf.reduce_mean(test_dist).numpy(),  tf.math.reduce_std(test_dist).numpy()/np.sqrt(test_dist.shape[0])))

Begin the 0th training epoch:


Train loss 5.7930e-03 +/- 2.1043e-04
 test loss 3.7121e-03 +/- 2.5459e-04
Begin the 1th training epoch:
Train loss 6.0374e-04 +/- 1.9382e-05
 test loss 2.9346e-04 +/- 1.7600e-05
Begin the 2th training epoch:
Train loss 2.5389e-04 +/- 7.3964e-06
 test loss 1.0731e-04 +/- 5.3260e-06
Begin the 3th training epoch:
Train loss 4.6962e-04 +/- 1.2111e-05
 test loss 4.3851e-04 +/- 2.0112e-05
Begin the 4th training epoch:
Train loss 1.1946e-04 +/- 3.2494e-06
 test loss 6.2774e-05 +/- 3.0838e-06
Begin the 5th training epoch:
Train loss 9.2691e-05 +/- 2.4717e-06
 test loss 4.8647e-05 +/- 2.1897e-06
Begin the 6th training epoch:
Train loss 1.3189e-04 +/- 3.7682e-06
 test loss 1.1600e-04 +/- 5.4034e-06
Begin the 7th training epoch:
Train loss 1.7194e-03 +/- 4.2064e-05
 test loss 1.6744e-03 +/- 7.5553e-05
Begin the 8th training epoch:
Train loss 4.6124e-05 +/- 1.3210e-06
 test loss 2.4432e-05 +/- 1.1440e-06
Begin the 9th training epoch:
Train loss 6.4813e-05 +/- 1.6498e-06
 test loss 4.2383e-05 +/- 1

In [2]:
model.save(config["path_model"])

NameError: name 'model' is not defined

In [ ]:
with tf.GradientTape() as tape:
    y_pred = model.get_gradient(train_states[1:2], separate_fields = config["separate_fields"])
    loss = loss_obj(train_timegrads[1:2],y_pred)
grads = tape.gradient(loss, model.trainable_variables)
optimizer.apply_gradients(zip(grads, model.trainable_variables))

<KerasVariable shape=(), dtype=int64, path=adam/iteration>

In [121]:
model.trainable_variables[0].numpy()

array([[-0.19177568,  0.87534827,  0.02920771,  0.2425765 ,  0.36881366],
       [-0.42200541, -0.00694296, -0.5405733 ,  0.52460206, -0.50546753]],
      dtype=float32)

In [122]:
grads

[<tf.Tensor: shape=(2, 5), dtype=float32, numpy=
 array([[ 0.        ,  0.3630526 ,  0.        , -0.05696164,  0.        ],
        [ 0.        ,  0.8020425 ,  0.        , -0.12583758,  0.        ]],
       dtype=float32)>,
 <tf.Tensor: shape=(5,), dtype=float32, numpy=array([0., 0., 0., 0., 0.], dtype=float32)>,
 <tf.Tensor: shape=(5, 5), dtype=float32, numpy=
 array([[ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
        [-0.500891  ,  0.19319718,  0.        , -0.6508339 ,  0.        ],
        [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
        [-0.811376  ,  0.3129534 ,  0.        , -1.0542634 ,  0.        ],
        [ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ]],
       dtype=float32)>,
 <tf.Tensor: shape=(5,), dtype=float32, numpy=array([0., 0., 0., 0., 0.], dtype=float32)>,
 <tf.Tensor: shape=(5, 5), dtype=float32, numpy=
 array([[-0.22802037,  0.        ,  0.        ,  0.16099705, -0.0213571 ],
        [-3.5375352 ,

In [111]:
2* (model.trainable_variables[0].numpy()[:,0:1] - train_timegrads[1:2].numpy().T)

array([[-4.5514526],
       [-7.643057 ]], dtype=float32)

In [112]:
train_timegrads[0:1].numpy()

array([[ 1.7640524, -0.       ]], dtype=float32)

In [113]:
model.nn(tf.ones_like(train_timegrads[1:2]))

<tf.Tensor: shape=(1, 2), dtype=float32, numpy=array([[-0.64867914,  0.5022094 ]], dtype=float32)>

In [123]:
def compute_numerical_gradients(model, x, y_true, epsilon=1e-5):
    numerical_gradients = []
    for param in model.trainable_variables:  # 遍历每个可训练参数
        grad = np.zeros_like(param.numpy())  # 初始化梯度数组
        param_value = param.numpy()  # 当前参数值

        # 遍历参数中的每个元素，计算数值梯度
        it = np.nditer(param_value, flags=['multi_index'], op_flags=['readwrite'])
        while not it.finished:
            idx = it.multi_index  # 获取当前元素的索引

            # f(param + epsilon)
            param_value[idx] += epsilon
            param.assign(param_value)  # 暂时更新参数
            loss_plus = loss_obj(y_true, model.get_gradient(x)).numpy()

            # f(param - epsilon)
            param_value[idx] -= 2 * epsilon
            param.assign(param_value)  # 暂时更新参数
            loss_minus = loss_obj(y_true, model.get_gradient(x)).numpy()

            # 恢复原始参数值
            param_value[idx] += epsilon
            param.assign(param_value)

            # 数值梯度公式
            grad[idx] = (loss_plus - loss_minus) / (2 * epsilon)
            it.iternext()
        
        numerical_gradients.append(grad)
    return numerical_gradients

In [124]:
num_grads = compute_numerical_gradients(model, train_states[1:2], train_timegrads[1:2])

In [126]:
for _g, _ng in zip(grads,num_grads):
    print(np.mean((_g-_ng)**2))

0.00247605
0.0
0.0016017343
0.0
0.008449017
0.0
0.004614354


In [59]:
import train
import importlib
importlib.reload(train)
from configs import CONFIGS

In [60]:
config = CONFIGS["LHNN_1DGaussianmixture"] # load the config which records all experiment parameters.
# data = gene_data.get_dataset(**config)
# print("Finished generating the dataset.")
model = train.train(config)

Successfully loaded data
Begin the 0th training epoch:


Train loss 5.9864e-03 +/- 2.0922e-04
 test loss 4.3878e-03 +/- 3.2016e-04
Begin the 1th training epoch:
Train loss 1.0524e-03 +/- 3.4995e-05
 test loss 4.9435e-04 +/- 2.5998e-05
Begin the 2th training epoch:
Train loss 8.5512e-04 +/- 2.3242e-05
 test loss 8.4770e-04 +/- 4.7371e-05
Begin the 3th training epoch:
Train loss 2.5171e-04 +/- 6.9894e-06
 test loss 1.8864e-04 +/- 9.4492e-06
Begin the 4th training epoch:
Train loss 4.6476e-04 +/- 1.1740e-05
 test loss 4.4487e-04 +/- 2.0658e-05
Begin the 5th training epoch:
Train loss 1.8653e-03 +/- 4.6979e-05
 test loss 1.8230e-03 +/- 8.6530e-05
Begin the 6th training epoch:
Train loss 7.2458e-05 +/- 2.2620e-06
 test loss 3.1889e-05 +/- 1.4392e-06
Begin the 7th training epoch:
Train loss 5.9839e-05 +/- 1.9074e-06
 test loss 2.4281e-05 +/- 1.1403e-06
Begin the 8th training epoch:
Train loss 5.0861e-05 +/- 1.6369e-06
 test loss 1.9480e-05 +/- 9.4271e-07
Begin the 9th training epoch:
Train loss 5.0374e-05 +/- 1.4881e-06
 test loss 2.3752e-05 +/- 9

In [61]:
data = gene_data.get_dataset(**config)

Successfully loaded data


In [62]:
# arrange data
train_states = tf.convert_to_tensor(data['train_states'], dtype=tf.float32)
test_states = tf.convert_to_tensor(data['test_states'], dtype=tf.float32)
train_timegrads = tf.convert_to_tensor(data['train_timegrads'], dtype=tf.float32)
test_timegrads = tf.convert_to_tensor(data['test_timegrads'], dtype=tf.float32)

In [63]:
train_timegrads_hat = model.get_gradient(train_states)
train_dist = (train_timegrads - train_timegrads_hat)**2 
test_timegrads_hat = model.get_gradient(test_states)
test_dist = (test_timegrads - test_timegrads_hat)**2
print('Train loss {:.4e} +/- {:.4e}\n test loss {:.4e} +/- {:.4e}'
    .format(tf.reduce_mean(train_dist).numpy(), tf.math.reduce_std(train_dist).numpy()/np.sqrt(train_dist.shape[0]),

            tf.reduce_mean(test_dist).numpy(),  tf.math.reduce_std(test_dist).numpy()/np.sqrt(test_dist.shape[0])))

Train loss 8.1526e-05 +/- 3.2732e-06
 test loss 1.1784e-05 +/- 4.2207e-07


In [64]:
weights = [va.numpy() for va in model.trainable_variables]

In [65]:
for _w in weights:
    print(_w)

[[ 2.45823413e-01  2.07824245e-01 -2.50091881e-01 -5.33072986e-02
   4.84837517e-02  7.63757452e-02 -1.22169413e-01  8.30326825e-02
  -1.61776468e-01 -2.36695170e-01 -1.51748195e-01  1.70040861e-01
   1.46442011e-01  1.39164329e-01  8.21178108e-02  2.66222298e-01
  -2.03978062e-01 -4.51216176e-02 -2.41365910e-01  1.56567767e-01
   2.28742257e-01  2.37346180e-02  2.48609960e-01  1.94126308e-01
  -1.96185052e-01  8.94899890e-02 -2.20420927e-01 -2.01779246e-01
   4.30059843e-02  7.70732388e-02 -1.35414958e-01 -2.05222681e-01
   2.50131711e-02 -2.10506216e-01 -1.34227976e-01  6.05700053e-02
   1.90544546e-01  1.57487348e-01  1.52928889e-01 -1.17538117e-01
   2.01223701e-01 -1.96746632e-01  8.29755813e-02  2.05874145e-01
   2.44862601e-01  1.09731510e-01  1.95284143e-01  1.23065211e-01
   1.39912903e-01  1.46127552e-01 -2.23695233e-01  2.58300006e-01
   1.73355162e-01  2.34421983e-01 -1.67864129e-01  2.71403402e-01
   2.69738525e-01  1.89967796e-01  2.45728835e-01 -2.80407876e-01
  -2.14324

In [66]:
model.save("mymodel.keras")


/home/ycui/.conda/envs/tf/lib/python3.10/site-packages/keras/src/saving/saving_api.py:107: UserWarning: You are saving a model that has not yet been built. It might not contain any weights yet. Consider building the model first by calling it on some data.
  return saving_lib.save_model(model, filepath)


In [86]:
model.save_weights("mymodel.weights.h5")

In [87]:
model2 = hnn.HNN(config["input_dim"], config["num_hidden"], config["num_layers"], config["output_dim"], acti=config["acti"], baseline=config["baseline"], field_type=config["field_type"])
model2.load_weights("mymodel.weights.h5")

In [80]:

# model_path = "/home/ycui/Documents/TSRLproject_JPMorgan/models/1DGaussianmixture.keras"
model_path = "mymodel.keras"
model1 = tf.keras.models.load_model(model_path)

In [88]:
def getloss(model):
    train_timegrads_hat = model.get_gradient(train_states)
    train_dist = (train_timegrads - train_timegrads_hat)**2 
    test_timegrads_hat = model.get_gradient(test_states)
    test_dist = (test_timegrads - test_timegrads_hat)**2
    print('Train loss {:.4e} +/- {:.4e}\n test loss {:.4e} +/- {:.4e}'
        .format(tf.reduce_mean(train_dist).numpy(), tf.math.reduce_std(train_dist).numpy()/np.sqrt(train_dist.shape[0]),
                tf.reduce_mean(test_dist).numpy(),  tf.math.reduce_std(test_dist).numpy()/np.sqrt(test_dist.shape[0])))

getloss(model2)

Train loss 8.1526e-05 +/- 3.2732e-06
 test loss 1.1784e-05 +/- 4.2207e-07


In [56]:
weights_new = [va.numpy() for va in model.trainable_variables]

In [58]:
for _w, _v in zip(weights, weights_new):
    print(np.sum(_w-_v))

0.0
0.0
0.0
0.0
0.0
0.0
0.0


In [77]:
model.hidden_layers[0](train_states) - model1.hidden_layers[0](train_states)

<tf.Tensor: shape=(6416, 100), dtype=float32, numpy=
array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], dtype=float32)>

In [78]:
model.call

<bound method HNN.call of <HNN name=hnn_2, built=False>>

In [79]:
model1.call

<bound method HNN.call of <HNN name=hnn_2, built=False>>

In [82]:
model1.call = model.call